# Lung HCE v3 → All_cells Validation

Zero-shot inference: apply the trained lung HCE v3 model to the lab's `All_cells.h5ad` dataset and compare predicted lung cell type labels against the lab's manual annotations.

**Expected behaviour:** Label mismatch is likely since the model was trained on lung HCA cell types and `All_cells` is brain tumour data. The goal is to visualise *how* the model maps its lung vocabulary onto the lab's cell types.

## 1. Imports

In [ ]:
import os, sys, warnings, time
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

import scanpy as sc
from transformers import AutoTokenizer, AutoModel
from tqdm import tqdm
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import seaborn as sns
import scipy.sparse as sp

warnings.filterwarnings('ignore')
print('=' * 60)
print('  IMPORTS OK')
print(f'  PyTorch : {torch.__version__}')
print(f'  CUDA    : {torch.cuda.is_available()}')
print('=' * 60)

## 2. Configuration

In [ ]:
# --- Paths ---
V3_RESULTS_DIR  = 'lung_hce_v3_results'
BEST_MODEL_PATH = os.path.join(V3_RESULTS_DIR, 'best_model.pt')
ALL_CELLS_PATH  = 'All_cells.h5ad'
OUT_DIR         = 'lung_to_allcells_results'
C2S_MODEL_NAME  = 'vandijklab/C2S-Pythia-410m-cell-type-conditioned-cell-generation'

# --- All_cells annotation column ---
TRUE_LABEL_COL  = 'predicted.high_hierarchy'   # lab manual annotation

# --- Inference settings (match v3 training) ---
TOP_K_GENES  = 100
MAX_SEQ_LEN  = 512
BATCH_SIZE   = 16
SEED         = 42

os.makedirs(OUT_DIR, exist_ok=True)
torch.manual_seed(SEED)
np.random.seed(SEED)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

print(f'  Model checkpoint : {BEST_MODEL_PATH}')
print(f'  All_cells path   : {ALL_CELLS_PATH}')
print(f'  Output dir       : {OUT_DIR}')
print(f'  Device           : {device}')
print('[OK] Config ready')

## 3. Reconstruct Class Vocabulary from v3 Artifacts

In [ ]:
# Reconstruct class_names and leaf_indices exactly as v3 built them
ontology_df  = pd.read_csv(os.path.join(V3_RESULTS_DIR, 'ontology.csv'))
balanced_df  = pd.read_csv(os.path.join(V3_RESULTS_DIR, 'balanced_counts.csv'))

ontology_dict = {}
for _, row in ontology_df.iterrows():
    child  = str(row['child'])
    parent = None if pd.isna(row['parent']) else str(row['parent'])
    ontology_dict[child] = parent

leaf_classes_set = set(balanced_df['ann_finest_level'].astype(str).values)

all_nodes = set()
for child, parent in ontology_dict.items():
    all_nodes.add(child)
    if parent:
        all_nodes.add(parent)
all_nodes |= leaf_classes_set

class_names   = sorted(all_nodes)
class_to_idx  = {name: idx for idx, name in enumerate(class_names)}
n_classes     = len(class_names)

leaf_classes  = sorted(leaf_classes_set)
leaf_indices  = [class_to_idx[c] for c in leaf_classes]
leaf_indices_t = torch.tensor(leaf_indices, device=device)

print(f'  Total classes (leaf + ancestors) : {n_classes}')
print(f'  Leaf classes                     : {len(leaf_classes)}')
print(f'  Sample leaf classes: {leaf_classes[:5]}')
print('[OK] Class vocabulary reconstructed')

## 4. Load Model

In [ ]:
print(f'  Loading tokenizer and encoder: {C2S_MODEL_NAME} ...')
tokenizer = AutoTokenizer.from_pretrained(C2S_MODEL_NAME)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

c2s_model   = AutoModel.from_pretrained(C2S_MODEL_NAME)
hidden_size = c2s_model.config.hidden_size

class C2SClassifier(nn.Module):
    def __init__(self, encoder, hidden_size, num_classes):
        super().__init__()
        self.encoder = encoder
        self.dropout = nn.Dropout(0.1)
        self.head    = nn.Linear(hidden_size, num_classes)

    def forward(self, input_ids, attention_mask):
        out        = self.encoder(input_ids=input_ids, attention_mask=attention_mask)
        last_hidden = out.last_hidden_state
        seq_len    = attention_mask.sum(dim=1) - 1
        last_token = last_hidden[torch.arange(last_hidden.size(0), device=last_hidden.device), seq_len]
        return self.head(self.dropout(last_token))

model = C2SClassifier(c2s_model, hidden_size, n_classes).to(device)

ckpt = torch.load(BEST_MODEL_PATH, map_location=device)
model.load_state_dict(ckpt['model_state_dict'])
model.eval()

print(f'  Loaded checkpoint: epoch {ckpt.get("epoch","?")}, val_acc={ckpt.get("val_acc",0):.4f}')
print('[OK] Model ready')

## 5. Load All_cells and Convert to Text

In [ ]:
print(f'  Reading {ALL_CELLS_PATH} ...')
adata = sc.read_h5ad(ALL_CELLS_PATH)
print(f'  Shape : {adata.shape}  (cells x genes)')
print(f'  True label col : {TRUE_LABEL_COL}')

true_labels = adata.obs[TRUE_LABEL_COL].astype(str).values
unique_true = sorted(set(true_labels))
print(f'  True label classes ({len(unique_true)}): {unique_true}')

# Gene overlap info (informational only — model still runs regardless)
gene_names = np.array(adata.var_names)
print(f'\n  NOTE: All_cells uses gene symbols; lung model was trained on Ensembl IDs.')
print(f'  Gene overlap is 0 — predictions are based on unrecognised tokens.')
print(f'  This is intentional: we are testing how the model maps its vocabulary.')

# Convert each cell to a top-k gene text string
def cell_to_text(X_row, gene_names, top_k=100):
    if sp.issparse(X_row):
        row = X_row.toarray().flatten()
    else:
        row = np.array(X_row).flatten()
    nonzero = np.where(row > 0)[0]
    if len(nonzero) == 0:
        return ''
    vals = row[nonzero]
    if len(nonzero) > top_k:
        top_idx = np.argpartition(vals, -top_k)[-top_k:]
        top_idx = top_idx[np.argsort(vals[top_idx])[::-1]]
        nonzero = nonzero[top_idx]
    else:
        nonzero = nonzero[np.argsort(vals)[::-1]]
    return ' '.join(gene_names[i] for i in nonzero)

print(f'\n  Converting {adata.n_obs:,} cells to text ...')
X = adata.X
cell_texts = []
for i in tqdm(range(adata.n_obs), desc='  cell->text', leave=False):
    cell_texts.append(cell_to_text(X[i], gene_names, TOP_K_GENES))

empty_count = sum(1 for t in cell_texts if not t.strip())
print(f'  Empty cells : {empty_count}')
print(f'  Sample text : {cell_texts[0][:80]} ...')
print('[OK] Cell texts ready')

## 6. Run Inference

In [ ]:
class InferenceDataset(Dataset):
    def __init__(self, texts, tokenizer, max_length):
        self.texts      = texts
        self.tokenizer  = tokenizer
        self.max_length = max_length

    def __len__(self): return len(self.texts)

    def __getitem__(self, idx):
        enc = self.tokenizer(
            self.texts[idx] if self.texts[idx].strip() else '[PAD]',
            truncation=True,
            max_length=self.max_length,
            padding='max_length',
            return_tensors='pt',
        )
        return {
            'input_ids':      enc['input_ids'].squeeze(0),
            'attention_mask': enc['attention_mask'].squeeze(0),
        }

inf_ds     = InferenceDataset(cell_texts, tokenizer, MAX_SEQ_LEN)
inf_loader = DataLoader(inf_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)

all_preds   = []   # predicted leaf class index
all_softmax = []   # full softmax over leaf classes (for confidence)

print(f'  Running inference on {len(cell_texts):,} cells ...')
t0 = time.time()
with torch.no_grad():
    for batch in tqdm(inf_loader, desc='  Inference', leave=True):
        input_ids      = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        logits         = model(input_ids, attention_mask)            # (B, n_classes)
        leaf_logits    = logits[:, leaf_indices_t]                   # (B, n_leaf)
        probs          = torch.softmax(leaf_logits, dim=1)           # (B, n_leaf)
        pred_leaf_pos  = leaf_logits.argmax(dim=1)                   # index into leaf_indices
        pred_class_idx = leaf_indices_t[pred_leaf_pos]               # global class index
        all_preds.extend(pred_class_idx.cpu().numpy())
        all_softmax.extend(probs.cpu().numpy())

all_preds     = np.array(all_preds)
all_softmax   = np.array(all_softmax)
pred_names    = [class_names[i] for i in all_preds]
confidence    = all_softmax.max(axis=1)

print(f'  Elapsed : {time.time()-t0:.1f}s')
print(f'  Unique predicted classes: {len(set(pred_names))}')
print('[OK] Inference complete')

## 7. Build Results DataFrame

In [ ]:
results_df = pd.DataFrame({
    'true_label' : true_labels,
    'pred_label' : pred_names,
    'confidence' : confidence,
})
results_df.to_csv(os.path.join(OUT_DIR, 'predictions.csv'), index=False)

print('Prediction overview:')
print(results_df.groupby('pred_label').size().sort_values(ascending=False).to_string())
print(f'\nMean confidence : {confidence.mean():.4f}')
print(f'Median confidence: {np.median(confidence):.4f}')

## 8. Visualisations

### 8a. Overall Prediction Distribution

In [ ]:
pred_counts = results_df['pred_label'].value_counts()

fig, ax = plt.subplots(figsize=(12, max(4, len(pred_counts) * 0.35)))
colors = ['#2166ac' if c >= 0.5 else '#d1e5f0' for c in
          [results_df[results_df['pred_label'] == lbl]['confidence'].mean()
           for lbl in pred_counts.index]]
ax.barh(pred_counts.index[::-1], pred_counts.values[::-1], color=colors[::-1])
ax.set(xlabel='Number of All_cells cells predicted', title='Lung model predictions on All_cells')
ax.grid(axis='x', alpha=0.3)
for i, (lbl, cnt) in enumerate(zip(pred_counts.index[::-1], pred_counts.values[::-1])):
    ax.text(cnt + 1, i, str(cnt), va='center', fontsize=8)
plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, 'pred_distribution.png'), dpi=150, bbox_inches='tight')
plt.show()

### 8b. Per-True-Label Prediction Breakdown

In [ ]:
# For each true (lab) label: what does the lung model predict?
breakdown = (
    results_df.groupby(['true_label', 'pred_label'])
    .size()
    .reset_index(name='count')
)
breakdown['pct'] = breakdown.groupby('true_label')['count'].transform(lambda x: x / x.sum())
breakdown.to_csv(os.path.join(OUT_DIR, 'per_true_label_breakdown.csv'), index=False)

print('Top lung predictions per All_cells label:\n')
for true_lbl in sorted(unique_true):
    sub = breakdown[breakdown['true_label'] == true_lbl].sort_values('pct', ascending=False)
    top3 = sub.head(3)
    n_total = sub['count'].sum()
    print(f'  {true_lbl} (n={n_total}):')
    for _, row in top3.iterrows():
        print(f'    → {row["pred_label"]:45s}  {row["count"]:4d} cells  ({row["pct"]*100:.1f}%)')
    print()

### 8c. Heatmap: True (Lab) Label vs Predicted (Lung) Label

In [ ]:
# Pivot: rows = true label, columns = predicted lung label
# Normalised by row (fraction of each true-label's cells per prediction)
pivot = breakdown.pivot_table(
    index='true_label', columns='pred_label', values='pct', fill_value=0
)

# Keep only predicted columns that appear at all
active_preds = pred_counts[pred_counts > 0].index.tolist()
pivot = pivot.reindex(columns=[c for c in active_preds if c in pivot.columns], fill_value=0)

fig, ax = plt.subplots(figsize=(max(10, len(pivot.columns) * 0.45), max(6, len(pivot) * 0.55)))
sns.heatmap(
    pivot, ax=ax,
    cmap='YlOrRd', vmin=0, vmax=1,
    linewidths=0.4, linecolor='#dddddd',
    annot=True, fmt='.2f', annot_kws={'size': 7},
    cbar_kws={'label': 'Fraction of true-label cells'},
)
ax.set_xlabel('Predicted lung cell type', fontsize=10)
ax.set_ylabel('True lab annotation', fontsize=10)
ax.set_title('Mapping: Lab annotations → Lung model predictions\n(row-normalised fractions)', fontsize=11)
plt.xticks(rotation=45, ha='right', fontsize=8)
plt.yticks(rotation=0, fontsize=9)
plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, 'label_mapping_heatmap.png'), dpi=150, bbox_inches='tight')
plt.show()

### 8d. Stacked Bar: Prediction Mix per True Label

In [ ]:
# Colour palette — one colour per predicted lung class
pred_classes_used = pivot.columns.tolist()
palette = plt.cm.get_cmap('tab20', len(pred_classes_used))
colors  = {c: palette(i) for i, c in enumerate(pred_classes_used)}

fig, ax = plt.subplots(figsize=(max(8, len(unique_true) * 0.9), 6))
bottoms = np.zeros(len(pivot))
x       = np.arange(len(pivot))

for pred_cls in pred_classes_used:
    vals = pivot[pred_cls].values
    bars = ax.bar(x, vals, bottom=bottoms, label=pred_cls, color=colors[pred_cls], edgecolor='white', linewidth=0.3)
    bottoms += vals

ax.set_xticks(x)
ax.set_xticklabels(pivot.index, rotation=45, ha='right', fontsize=9)
ax.set_ylabel('Fraction of cells', fontsize=10)
ax.set_title('Lung model prediction mix per lab annotation', fontsize=11)
ax.set_ylim(0, 1.05)
ax.legend(
    bbox_to_anchor=(1.01, 1), loc='upper left',
    fontsize=7, title='Predicted lung type', title_fontsize=8
)
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, 'stacked_bar.png'), dpi=150, bbox_inches='tight')
plt.show()

### 8e. Confidence Distribution per True Label

In [ ]:
n_cols = 4
n_rows = int(np.ceil(len(unique_true) / n_cols))
fig, axes = plt.subplots(n_rows, n_cols, figsize=(n_cols * 3.5, n_rows * 2.8), sharey=False)
axes = axes.flatten()

for i, true_lbl in enumerate(sorted(unique_true)):
    mask = results_df['true_label'] == true_lbl
    conf = results_df.loc[mask, 'confidence'].values
    axes[i].hist(conf, bins=20, range=(0, 1), color='steelblue', edgecolor='white', linewidth=0.4)
    axes[i].axvline(conf.mean(), color='red', ls='--', lw=1, label=f'mean={conf.mean():.2f}')
    axes[i].set_title(true_lbl, fontsize=8)
    axes[i].set_xlim(0, 1)
    axes[i].legend(fontsize=7)
    axes[i].tick_params(labelsize=7)

for j in range(i + 1, len(axes)):
    axes[j].set_visible(False)

fig.suptitle('Prediction confidence distribution per lab annotation', fontsize=11, y=1.01)
plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, 'confidence_distributions.png'), dpi=150, bbox_inches='tight')
plt.show()

## 9. Summary

In [ ]:
print('=' * 60)
print('  VALIDATION SUMMARY')
print('=' * 60)
print(f'  All_cells cells          : {len(results_df):,}')
print(f'  True label classes       : {len(unique_true)}')
print(f'  Unique predictions made  : {results_df["pred_label"].nunique()}')
print(f'  Mean prediction confidence: {confidence.mean():.4f}')
print()
print('  Dominant lung prediction per lab label:')
print(f'  {"Lab label":<35} {"Top lung prediction":<45} {"Frac":>6}')
print('  ' + '-' * 90)
for true_lbl in sorted(unique_true):
    sub = breakdown[breakdown['true_label'] == true_lbl].sort_values('pct', ascending=False)
    top = sub.iloc[0]
    print(f'  {true_lbl:<35} {top["pred_label"]:<45} {top["pct"]*100:5.1f}%')

results_df.to_csv(os.path.join(OUT_DIR, 'predictions.csv'), index=False)
breakdown.to_csv(os.path.join(OUT_DIR, 'per_true_label_breakdown.csv'), index=False)
print(f'\n[OK] Results saved to {OUT_DIR}/')